In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from skimage.color import rgb2gray
from skimage.feature import hog

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import torchvision
import torchvision.transforms as transforms

Load CIFAR-10

In [ ]:
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="../data",
    train=False,
    download=False,
    transform=transform
)

classes = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

Convert one tensor image to NumPy

In [ ]:
image, label = train_dataset[0]

image_np = image.permute(1, 2, 0).numpy()

print("Original tensor shape:", image.shape)
print("NumPy image shape:", image_np.shape)
print("Class:", classes[label])

plt.imshow(image_np)
plt.title(classes[label])
plt.axis("off")
plt.show()

Convert RGB image to grayscale

In [ ]:
gray_image = rgb2gray(image_np)

print("Grayscale shape:", gray_image.shape)

plt.imshow(gray_image, cmap="gray")
plt.title("Grayscale Image")
plt.axis("off")
plt.show()

Extract HOG features

In [ ]:
hog_features, hog_image = hog(
    gray_image,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    visualize=True,
    feature_vector=True
)

print("Number of HOG features:", len(hog_features))

Visualize original and HOG

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

axes[0].imshow(image_np)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(hog_image, cmap="gray")
axes[1].set_title("HOG Representation")
axes[1].axis("off")

plt.tight_layout()
plt.show()

Function for extracting HOG

In [ ]:
def extract_hog_features(dataset, max_samples=None):
    features = []
    labels = []

    total = len(dataset)

    if max_samples is not None:
        total = min(max_samples, total)

    for i in range(total):
        image, label = dataset[i]

        image_np = image.permute(1, 2, 0).numpy()
        gray = rgb2gray(image_np)

        feature_vector = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            feature_vector=True
        )

        features.append(feature_vector)
        labels.append(label)

    return np.array(features), np.array(labels)

Extract a smaller subset first

In [ ]:
X_train, y_train = extract_hog_features(
    train_dataset,
    max_samples=10000
)

X_test, y_test = extract_hog_features(
    test_dataset,
    max_samples=2000
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

Train SVM

In [ ]:
svm_model = SVC(
    kernel="rbf",
    C=10,
    gamma="scale"
)

svm_model.fit(X_train, y_train)

Prediction

In [ ]:
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("HOG + SVM Accuracy:", accuracy)

Classification report

In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=classes
    )
)

Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(9, 7))
plt.imshow(cm)
plt.title("HOG + SVM Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks(range(10), classes, rotation=45)
plt.yticks(range(10), classes)

plt.colorbar()
plt.tight_layout()
plt.show()

In [ ]:
import joblib

joblib.dump(svm_model, "../results/hog_svm_cifar10.pkl")

print("SVM model saved.")